# Deriving the 2026-09 deposits


Every column added in the September 2026 data audit comes from a public deposit through a computation -- a mean of replicate screens, a log, a moderated contrast. This notebook runs each one from the raw file and shows the check that admitted it. The code is `starplast/deposits.py`; this notebook calls it, so what is shown is what ships.

Raw deposits live under the dataset root, in the taxonomy `<level>/<kind>/<PMID or accession>/`; every folder carries `URLS.txt` and `SHA256SUMS.txt` from the download.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from scipy import stats
ROOT = '/media/carruthers/mnt3/claude/repo/starplast'
DATA = '/media/carruthers/mnt3/claude/toxoplasma_projects/datasets'
import sys; sys.path.insert(0, ROOT)
from starplast import deposits as D
nodes = pd.read_parquet(os.path.join(ROOT, 'starplast', 'data', 'nodes.parquet'))
product = nodes.set_index('gene_id')['product']
ribosomal = product.str.contains('ribosomal protein', case=False, na=False) & ~product.str.contains('mitochondrial|apicoplast|kinase|methyltransferase|acetyltransferase', case=False, na=False)
def me49(ids): return 'TGME49_' + pd.Series(ids, dtype=str).str.extract(r'_(\d{6})')[0]
print(len(nodes), 'genes;', int(ribosomal.sum()), 'cytosolic ribosomal proteins')

8140 genes; 155 cytosolic ribosomal proteins


## 1. In vivo fitness in six tissues (Giuliano et al. 2024, PMID 38977907)

The four tissues already shipped were cited to the 2019 platform paper. If they are this study's composite scores, the supplement must reproduce them value for value. It does, and it carries heart and brain as well.

In [2]:
invivo = D.giuliano_invivo(DATA)
invivo['me49'] = me49(invivo.gene_id).values
whole = invivo[~invivo.gene_id.str.contains(r'\d[A-Z]$')].drop_duplicates('me49').set_index('me49')
rows = []
for c in ['fit_invivo_PE', 'fit_invivo_liver', 'fit_invivo_spleen', 'fit_invivo_lung']:
    j = nodes.set_index('gene_id')[[c]].join(whole[[c]], rsuffix='_supp').dropna()
    rows.append({'column': c, 'genes': len(j), 'max |difference|': (j[c] - j[c + '_supp']).abs().max(),
                 'spearman': stats.spearmanr(j[c], j[c + '_supp']).correlation})
pd.DataFrame(rows)

,column,genes,max |difference|,spearman
0,fit_invivo_PE,7395,4.977e-08,1
1,fit_invivo_liver,7395,4.939e-08,1
2,fit_invivo_spleen,7395,4.878e-08,1
3,fit_invivo_lung,7395,4.99e-08,1


In [3]:
pd.DataFrame({t: invivo[f'fit_invivo_{t}'].describe() for t in ['PE', 'lung', 'heart', 'brain']}).round(2)

,PE,lung,heart,brain
count,8332,8332,8332,8332
mean,-2.89,-5.95,-3.97,-1.79
std,33.83,27.94,6.83,3.6
min,-798.9,-835.7,-79.9,-31.65
25%,-1.53,-3.61,-5.53,-2.77
50%,-0.05,-0.3,-1.5,-0.68
75%,0.11,0.02,-0.07,-0.01
max,515.6,410.6,73.6,38.32


## 2. Serum restriction (Bitew et al. 2025, PMID 41407671)

Two independent genome-wide screens, each in 10% and 1% serum. A screen of fibroblast fitness must find the ribosome essential; the difference between sera must NOT be fibroblast fitness again, or it is no new axis. GRA38 is the paper's gene.

In [4]:
serum = D.serum_restriction(DATA)
serum['me49'] = me49(serum.gene_id).values
s = serum[~serum.gene_id.str.contains(r'\d[A-Z]$')].drop_duplicates('me49').set_index('me49')
s = s.join(nodes.set_index('gene_id')[['fit_invitro_hff']])
rib = ribosomal.reindex(s.index).fillna(False).astype(bool)
pd.DataFrame({c: {'ribosomal median': s.loc[rib, c].median(), 'others median': s.loc[~rib, c].median(),
                  'rho with fit_invitro_hff': stats.spearmanr(s[c], s.fit_invitro_hff, nan_policy='omit').correlation}
              for c in s.columns if c.startswith('fit_') and c != 'fit_invitro_hff'}).T.round(3)

,ribosomal median,others median,rho with fit_invitro_hff
fit_lipid_rich_p8,-6.353,-3.119,0.847
fit_lipid_limited_p8,-6.126,-2.823,0.847
fit_lipid_rich_p4p5,-5.176,-1.397,0.842
fit_lipid_limited_p4p5,-5.253,-1.291,0.844
fit_serum_differential_p8,-0.201,-0.21,0.064
fit_serum_differential_p4p5,0.139,0.034,-0.209


In [5]:
serum.set_index('gene_id').loc[['TGGT1_312420']].T  # GRA38

gene_id,TGGT1_312420
fit_lipid_rich_p8,-3.637
fit_lipid_limited_p8,1.491
fit_lipid_rich_p4p5,1.159
fit_lipid_limited_p4p5,2.223
fit_serum_differential_p8,-5.128
fit_serum_differential_p4p5,-1.064
me49,TGME49_312420


## 3. Translation efficiency, GSE302107

The authors' TE is a linear ratio of footprint RPKM to RNA RPKM; it is shipped as log2 to sit on the scale of the other TE columns. The test is reproducibility, agreement with the earlier studies, and the ribosome being the high-TE class.

In [6]:
te = D.riboseq_302107(DATA).set_index('gene_id')
other = nodes.set_index('gene_id').filter(regex='^te(99395_intracellular|245775_parent_tachy)')
{'replicate rho': stats.spearmanr(te.te302107_tachy_r1, te.te302107_tachy_r2, nan_policy='omit').correlation,
 'rho with GSE99395 mean': stats.spearmanr(te.mean(axis=1), other.filter(like='99395').mean(axis=1).reindex(te.index), nan_policy='omit').correlation,
 'rho with GSE245775 mean': stats.spearmanr(te.mean(axis=1), other.filter(like='245775').mean(axis=1).reindex(te.index), nan_policy='omit').correlation,
 'ribosomal median log2 TE': te.mean(axis=1)[ribosomal.reindex(te.index).fillna(False).astype(bool)].median(),
 'others median log2 TE': te.mean(axis=1)[~ribosomal.reindex(te.index).fillna(False).astype(bool)].median()}

{'replicate rho': np.float64(0.9717565573467376), 'rho with GSE99395 mean': np.float64(0.7487017938696973), 'rho with GSE245775 mean': np.float64(0.7494289059739437), 'ribosomal median log2 TE': np.float64(1.3644291234578687), 'others median log2 TE': np.float64(0.1144645420467956)}

## 4. mRNA decay after actinomycin D, GSE329845

Raw counts, no spike-in: the result is decay relative to the median transcript. The model is intercept + treatment + replicate, moderated as in limma. Replicate agreement is checked on the per-replicate paired log ratios; stable ribosomal mRNAs are the biology check.

In [7]:
decay, fit = D.mrna_decay_329845(DATA, return_fit=True)
norm = fit['normalized']
paired = pd.DataFrame({i: np.log2(norm[f'cWT_ActD_REP{i}'] + 1) - np.log2(norm[f'cWT_vehicle_REP{i}'] + 1) for i in (1, 2, 3)}).loc[decay.gene_id]
print('size factors', {k: round(v, 3) for k, v in fit['size_factors'].items()})
print('prior df', round(fit['df_prior'], 1))
paired.corr(method='spearman').round(3)

size factors {'cWT_vehicle_REP1': 3.142, 'cMUT_vehicle_REP1': 4.156, 'cWT_vehicle_REP2': 3.139, 'cMUT_vehicle_REP2': 2.62, 'cWT_vehicle_REP3': 3.226, 'cMUT_vehicle_REP3': 3.029, 'cWT_ActD_REP1': 0.39, 'cMUT_ActD_REP1': 0.411, 'cWT_ActD_REP2': 0.292, 'cMUT_ActD_REP2': 0.245, 'cWT_ActD_REP3': 0.299, 'cMUT_ActD_REP3': 0.309}
prior df 4.4


,1,2,3
1,1,0.966,0.963
2,0.966,1,0.961
3,0.963,0.961,1


In [8]:
d = decay.assign(me49=me49(decay.gene_id).values).set_index('me49')['mrna_log2_remaining_4h_actinomycin']
rib = ribosomal.reindex(d.index).fillna(False).astype(bool)
old = nodes.set_index('gene_id')['mrna_remaining_5h_actinomycin'].reindex(d.index)
{'genes': len(d), 'ribosomal median': d[rib].median(), 'others median': d[~rib].median(),
 'Mann-Whitney p': stats.mannwhitneyu(d[rib], d[~rib]).pvalue,
 'rho with the shipped 412-gene column': stats.spearmanr(d, old, nan_policy='omit').correlation,
 'shared genes': int(old.notna().sum())}

{'genes': 6406, 'ribosomal median': np.float64(1.4255177589843855), 'others median': np.float64(-0.1696065040180186), 'Mann-Whitney p': np.float64(3.2732681637830885e-14), 'rho with the shipped 412-gene column': np.float64(-0.02104841441400259), 'shared genes': 298}

## 5. Host responses and a baseline macrophage

Host tables are keyed by reviewed UniProt accession through Ensembl ids (`reference/uniprot`), never by symbol. Each contrast is infected against uninfected with the design's blocks; the checks are genes whose behaviour is not in doubt.

In [9]:
hff = D.hff_tg_infection(DATA).drop_duplicates('host_name').set_index('host_name')
print(int((hff.hff_tg_infection_padj < 0.05).sum()), 'genes at padj < 0.05 of', len(hff))
hff.reindex(['CXCL8', 'IL6', 'CXCL10', 'ISG15', 'EGR1', 'GAPDH', 'ACTB']).round(3)

315 genes at padj < 0.05 of 10630


,hff_tg_infection_log2fc,hff_tg_infection_padj,host_id
host_name,,,
CXCL8,5.353,0,P10145
IL6,2.666,0,P05231
CXCL10,2.262,0.004,P02778
ISG15,2.134,0.004,P05161
EGR1,1.236,0.029,P18146
GAPDH,-0.025,0.871,P04406
ACTB,-0.028,0.887,P60709


In [10]:
bmdm = D.bmdm_baseline(DATA).drop_duplicates('host_name').set_index('host_name')
bmdm['rank'] = bmdm.bmdm_tpm.rank(ascending=False).astype(int)
bmdm.reindex(['Lyz2', 'Cd68', 'Csf1r', 'Emr1', 'Itgam', 'Alb', 'Apoa1']).round(1)

,bmdm_tpm,host_id,rank
host_name,,,
Lyz2,2.75e+04,P08905,1
Cd68,2791,P31996,39
Csf1r,1276,P09581,123
Emr1,903.2,Q61549,164
Itgam,NaN,NaN,NaN
Alb,0,P07724,1.414e+04
Apoa1,0,Q00623,1.414e+04


In [11]:
hep = D.hepatocyte_pf_infection(DATA).drop_duplicates('host_name').set_index('host_name')
print(int((hep.hepatocyte_pf_infection_padj < 0.05).sum()), 'genes at padj < 0.05 of', len(hep))
hep.sort_values('hepatocyte_pf_infection_log2fc', ascending=False).head(10).round(3)

25 genes at padj < 0.05 of 12550


,hepatocyte_pf_infection_log2fc,hepatocyte_pf_infection_padj,host_id
host_name,,,
IFI44L,4.481,0.077,Q53G44
CXCL10,2.674,0.076,P02778
RSAD2,2.336,0.083,Q8WXG1
CXCL11,2.284,0.025,O14625
MX2,2.264,0.092,P20592
CTHRC1,2.187,0.006,Q96CG8
SFRP1,2.178,0.134,Q8N474
OASL,2.171,0.006,Q15646
HGF,2.021,0.116,P14210


## 6. Carbon-source withdrawal (Uboldi et al., bioRxiv 2025)

The dependence column is the authors' contrast, glutamine-only minus glucose-only: negative means the gene is needed when glucose is absent. The genes of glutamine catabolism must come first, and the ribosome must be needed in complete medium.

In [12]:
carbon = D.glucose_limitation(DATA)
carbon['rank'] = carbon.fit_glucose_dependence.rank()
known = {'TGGT1_249390': 'GDH1', 'TGGT1_289650': 'PEPCK'}
print(carbon.set_index('gene_id').loc[list(known), ['fit_glucose_dependence', 'fit_glucose_dependence_fdr', 'rank']].rename(index=known).round(3))
c = carbon.assign(me49=me49(carbon.gene_id).values).drop_duplicates('me49').set_index('me49')
rib = ribosomal.reindex(c.index).fillna(False).astype(bool)
{'ribosomal median, complete medium': c.loc[rib, 'fit_complete_medium_2025'].median(),
 'others median, complete medium': c.loc[~rib, 'fit_complete_medium_2025'].median(),
 'genes at FDR 0.05': int((carbon.fit_glucose_dependence_fdr < 0.05).sum())}

         fit_glucose_dependence  fit_glucose_dependence_fdr  rank
gene_id                                                          
GDH1                     -9.989                         0.0   1.0
PEPCK                    -9.290                         0.0   2.0


{'ribosomal median, complete medium': np.float64(-2.31804479688773), 'others median, complete medium': np.float64(0.02669186048963055), 'genes at FDR 0.05': 175}

## 7. 5' UTR architecture (Peters et al., bioRxiv 2025)

Sequence features, admitted because they predict translation the way they must: more upstream AUGs, less translation; a better start context, more.

In [13]:
utr = D.utr5_architecture(DATA).set_index('gene_id')
te_mean = nodes.set_index('gene_id').filter(regex=r'^te\d').mean(axis=1)
{c: stats.spearmanr(utr[c], te_mean.reindex(utr.index), nan_policy='omit').correlation
 for c in ['utr5_n_uaugs', 'utr5_n_uorfs', 'utr5_length', 'utr5_kozak_score']}

{'utr5_n_uaugs': np.float64(-0.46611677979608285), 'utr5_n_uorfs': np.float64(-0.4005041747152712), 'utr5_length': np.float64(-0.3760310058221192), 'utr5_kozak_score': np.float64(0.21667402264403923)}

## 8. Bradyzoite subtypes in the brain (Ulu et al. 2026)

Five groups, all of them bradyzoites: the bradyzoite markers high everywhere, the tachyzoite antigen SAG1 low everywhere, and SRS22A marking Group B.

In [14]:
bz = D.bradyzoite_subtypes(DATA).set_index('gene_id')
pct = bz.rank(pct=True)
pct.loc[['TGME49_259020', 'TGME49_291040', 'TGME49_268860', 'TGME49_233460']].rename(index={'TGME49_259020': 'BAG1', 'TGME49_291040': 'LDH2', 'TGME49_268860': 'ENO1', 'TGME49_233460': 'SAG1'}).round(2)

,bzsub_A_expr,bzsub_B_expr,bzsub_C_expr,bzsub_D_expr,bzsub_E_expr
gene_id,,,,,
BAG1,1,0.98,1,1,1
LDH2,1,1,1,1,0.99
ENO1,0.99,0.98,1,0.99,0.97
SAG1,0.1,0.41,0.09,0.3,0.17


## 9. Iron depletion (Hanna et al., mBio 2026)

Iron-sulfur proteins need the iron that was withdrawn, so they should shift down -- and they do, modestly; protein and transcript should move together, though not in lockstep.

In [15]:
iron = D.iron_depletion(DATA)
iron['me49'] = me49(iron.gene_id).values
# Proteins arrive on GT1 accessions and transcripts on ME49 ones; pair them by gene.
i = iron.groupby('me49')[['iron_depletion_protein_log2fc', 'iron_depletion_rna_log2fc']].first()
# The paper's own iron-sulfur flag, rather than a guess from product names.
s1 = pd.read_excel(os.path.join(DATA, *D.IRON, 'mbio.03788-25-s0002.xlsx'), sheet_name=0, header=1)
fes = set(me49(s1.loc[s1['FeS'].notna(), 'Protein_Accessions']).dropna())
flag = i.index.isin(fes)
{'iron-sulfur proteins': int(flag.sum()),
 'iron-sulfur median log2fc': i.loc[flag, 'iron_depletion_protein_log2fc'].median(),
 'others median log2fc': i.loc[~flag, 'iron_depletion_protein_log2fc'].median(),
 'protein vs RNA rho': stats.spearmanr(i.iron_depletion_protein_log2fc, i.iron_depletion_rna_log2fc, nan_policy='omit').correlation}

{'iron-sulfur proteins': 61, 'iron-sulfur median log2fc': np.float64(-0.0432), 'others median log2fc': np.float64(0.0188), 'protein vs RNA rho': np.float64(0.38238432183511234)}

## 10. Organelle surfaces (Parker & Huet, bioRxiv 2026)

A bait on the outside of the mitochondrion should find mitochondrial proteins, and one on the ER should find ER proteins -- judged against hyperLOPIT, which measured location independently. The apicoplast bait is the weak one, and the notebook shows it.

In [16]:
surf = D.organelle_surface(DATA).set_index('gene_id')
comp = nodes.set_index('gene_id')['compartment'].reindex(surf.index)
rows = []
for bait, words in (('mitochondrion', 'mitochondri'), ('er', 'ER'), ('apicoplast', 'apicoplast')):
    seen = surf[f'surface_{bait}_stringent'].notna() & comp.notna()
    hit = surf[f'surface_{bait}_stringent'] == 1
    lab = comp.astype(str).str.contains(words)
    table = [[int((seen & hit & lab).sum()), int((seen & hit & ~lab).sum())],
             [int((seen & ~hit & lab).sum()), int((seen & ~hit & ~lab).sum())]]
    odds, p = stats.fisher_exact(table)
    rows.append({'bait': bait, 'stringent hits': int(hit.sum()), 'odds ratio': odds, 'p': p})
pd.DataFrame(rows)

,bait,stringent hits,odds ratio,p
0,mitochondrion,40,13.77,5.407e-12
1,er,44,7.597,8.831e-10
2,apicoplast,120,1.158,0.8228


## 11. Host genes rhoptry discharge needs (Valleau et al., bioRxiv 2025)

The screen must recover its own pathway: SLC35A2 first, the N-glycan genes near the top, and the glycosylation genes the paper rules out nowhere near it.

In [17]:
k562 = D.k562_rhoptry_screen(DATA).drop_duplicates('host_name').set_index('host_name')
rank = k562.rhoptry_discharge_score.rank(ascending=False)
rank.reindex(['SLC35A2', 'RFT1', 'DPAGT1', 'GFPT1', 'MGAT1', 'MGAT2', 'B4GALT1', 'FUT8', 'ST6GAL1']).astype('Int64')

host_name
SLC35A2        1
RFT1           9
DPAGT1        50
GFPT1         57
MGAT1        385
MGAT2        284
B4GALT1     8699
FUT8        7671
ST6GAL1    18535

## 12. Plasmodium: melting temperature, fertility, and the third wave

Melting points are fitted by `scripts/fit_meltome.py`, since none is published. The checks: chaperones labile and glycolysis stable; HAP2 needed by males only; and the counts the papers report, reproduced.

In [18]:
pf = pd.read_parquet(os.path.join(ROOT, 'starplast', 'data', 'pf_nodes.parquet')).set_index('gene_id')
tm = D.pf_melting_temperature(DATA).set_index('gene_id')['melting_temperature_tm']
prod = pf['product'].reindex(tm.index).fillna('')
{'HSP70/90 median Tm': tm[prod.str.contains('heat shock protein 70|heat shock protein 90', case=False)].median(),
 'glycolytic median Tm': tm[prod.str.contains('glyceraldehyde-3-phosphate|enolase|pyruvate kinase', case=False)].median(),
 'proteins': len(tm)}

{'HSP70/90 median Tm': np.float64(51.02895261810649), 'glycolytic median Tm': np.float64(61.89682454822203), 'proteins': 2040}

In [19]:
D.pb_fertility(DATA).set_index('gene_id').loc[['PF3D7_1014200']].rename(index={'PF3D7_1014200': 'HAP2'})

,fertility_female,fertility_male
gene_id,,
HAP2,0.2102,-8.071


In [20]:
{'gametocyte proteins newly made (paper: 705)': int(D.pf_gametocyte_proteome(DATA).gametocyte_newly_made.sum()),
 'latency classifier genes (paper: 200)': int(D.pf_latency(DATA).latency_classifier_member.sum()),
 'H3K27ac high-confidence (paper: 99)': int(D.pf_chromatin_proxiome(DATA).chromprox_h3k27ac_hit.sum()),
 'H3K4me3 high-confidence (paper: 48)': int(D.pf_chromatin_proxiome(DATA).chromprox_h3k4me3_hit.sum()),
 'febrile unique sites up / down (paper rows: 143 / 53)': tuple(int(x) for x in D.pf_febrile_phospho(DATA)[['febrile_phospho_n_sites_up', 'febrile_phospho_n_sites_down']].sum()),
 'm6A transcripts with a site': int((D.pf_m6a(DATA).m6a_n_canonical_sites > 0).sum()),
 'proteins engaged by at least one antimalarial': int((D.pf_target_engagement(DATA).engaged_n_compounds_hit > 0).sum())}

{'gametocyte proteins newly made (paper: 705)': 705, 'latency classifier genes (paper: 200)': 200, 'H3K27ac high-confidence (paper: 99)': 99, 'H3K4me3 high-confidence (paper: 48)': 48, 'febrile unique sites up / down (paper rows: 143 / 53)': (128, 51), 'm6A transcripts with a site': 3911, 'proteins engaged by at least one antimalarial': 175}

## 13. Write the tables

Each derivation is written to `starplast/data/deposit_<key>.tsv`. `scripts/add_deposits.py` merges them into the node and host tables, refusing any merge that would lose a value.

In [21]:
written = D.derive_all(DATA, ROOT, log=print)
{k: v.shape for k, v in written.items()}

deposit crispr_invivo_composite: 8,332 rows x 6 columns
deposit crispr_serum_restriction: 8,158 rows x 6 columns
deposit crispr_glucose_limitation: 8,155 rows x 5 columns
deposit gse302107_riboseq: 5,992 rows x 2 columns
deposit gse302108_utr5: 5,992 rows x 6 columns
deposit mrna_decay_gse329845: 6,406 rows x 1 columns
deposit brady_subtypes: 8,920 rows x 5 columns
deposit iron_depletion_proteome: 8,160 rows x 3 columns
deposit organelle_surface_turboid: 742 rows x 6 columns
deposit pf_meltome: 2,040 rows x 2 columns
deposit pf_pb_fertility_transfer: 1,121 rows x 2 columns
deposit pf_latency_transcriptome: 4,887 rows x 2 columns
deposit pf_m6a_nanopore: 5,285 rows x 2 columns
deposit pf_gametocyte_proteome: 2,544 rows x 2 columns
deposit pf_target_engagement: 3,126 rows x 2 columns
deposit pf_febrile_phospho: 1,874 rows x 3 columns
deposit pf_chromatin_proxiome: 2,020 rows x 8 columns
deposit host_hff_tg_infection: 10,631 rows x 3 columns
deposit host_bmdm_baseline: 15,437 rows x 2 col

{'crispr_invivo_composite': (8332, 7), 'crispr_serum_restriction': (8158, 7), 'crispr_glucose_limitation': (8155, 6), 'gse302107_riboseq': (5992, 3), 'gse302108_utr5': (5992, 7), 'mrna_decay_gse329845': (6406, 2), 'brady_subtypes': (8920, 6), 'iron_depletion_proteome': (8160, 4), 'organelle_surface_turboid': (742, 7), 'pf_meltome': (2040, 3), 'pf_pb_fertility_transfer': (1121, 3), 'pf_latency_transcriptome': (4887, 3), 'pf_m6a_nanopore': (5285, 3), 'pf_gametocyte_proteome': (2544, 3), 'pf_target_engagement': (3126, 3), 'pf_febrile_phospho': (1874, 4), 'pf_chromatin_proxiome': (2020, 9), 'host_hff_tg_infection': (10631, 4), 'host_bmdm_baseline': (15437, 3), 'host_hepatocyte_pf_infection': (12550, 4), 'host_k562_rhoptry_screen': (18739, 5)}